In [6]:
import pandas as pd


In [7]:
df = pd.read_excel(r'C:\Users\User\OneDrive\Documents\customer_shopping_behavior.csv.xlsx')


In [8]:
df.head(5)

,Unnamed: 0,Age,Gender,Item Purchased,Category,Purchase Amount (USD),Location,Size,Color,Season,Review Rating,Subscription Status,Shipping Type,Discount Applied,Promo Code Used,Previous Purchases,Payment Method,Frequency of Purchases
0,1,55,Male,Blouse,Clothing,53,Kentucky,L,Gray,Winter,3.1,Yes,Express,Yes,Yes,14,Venmo,Fortnightly
1,2,19,Male,Sweater,Clothing,64,Maine,L,Maroon,Winter,3.1,Yes,Express,Yes,Yes,2,Cash,Fortnightly
2,3,50,Male,Jeans,Clothing,73,Massachusetts,S,Maroon,Spring,3.1,Yes,Free Shipping,Yes,Yes,23,Credit Card,Weekly
3,4,21,Male,Sandals,Footwear,90,Rhode Island,M,Maroon,Spring,3.5,Yes,Next Day Air,Yes,Yes,49,PayPal,Weekly
4,5,45,Male,Blouse,Clothing,49,Oregon,M,Turquoise,Spring,2.7,Yes,Free Shipping,Yes,Yes,31,PayPal,Annually


In [9]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3900 entries, 0 to 3899
Data columns (total 18 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Unnamed: 0              3900 non-null   int64  
 1   Age                     3900 non-null   int64  
 2   Gender                  3900 non-null   object 
 3   Item Purchased          3900 non-null   object 
 4   Category                3900 non-null   object 
 5   Purchase Amount (USD)   3900 non-null   int64  
 6   Location                3900 non-null   object 
 7   Size                    3900 non-null   object 
 8   Color                   3900 non-null   object 
 9   Season                  3900 non-null   object 
 10  Review Rating           3863 non-null   float64
 11  Subscription Status     3900 non-null   object 
 12  Shipping Type           3900 non-null   object 
 13  Discount Applied        3900 non-null   object 
 14  Promo Code Used         3900 non-null   

In [10]:
#we check summarry statistivs
df.describe()

,Unnamed: 0,Age,Purchase Amount (USD),Review Rating,Previous Purchases
count,3900.000000,3900.000000,3900.000000,3863.000000,3900.000000
mean,1950.500000,44.068462,59.764359,3.750065,25.351538
std,1125.977353,15.207589,23.685392,0.716983,14.447125
min,1.000000,18.000000,20.000000,2.500000,1.000000
25%,975.750000,31.000000,39.000000,3.100000,13.000000
50%,1950.500000,44.000000,60.000000,3.800000,25.000000
75%,2925.250000,57.000000,81.000000,4.400000,38.000000
max,3900.000000,70.000000,100.000000,5.000000,50.000000


In [11]:
#we cehck on missing values
df.isnull().sum()

Unnamed: 0                 0
Age                        0
Gender                     0
Item Purchased             0
Category                   0
Purchase Amount (USD)      0
Location                   0
Size                       0
Color                      0
Season                     0
Review Rating             37
Subscription Status        0
Shipping Type              0
Discount Applied           0
Promo Code Used            0
Previous Purchases         0
Payment Method             0
Frequency of Purchases     0
dtype: int64

In [12]:
df['Review Rating'] = df.groupby('Category')['Review Rating'].transform(lambda x : x.fillna(x.median()))

In [13]:
#we now ensure uniformity with the our column names for ease of reference
df.columns = df.columns.str.lower()
df.columns = df.columns.str.replace(' ', '_')

In [14]:
df.columns

Index(['unnamed:_0', 'age', 'gender', 'item_purchased', 'category',
       'purchase_amount_(usd)', 'location', 'size', 'color', 'season',
       'review_rating', 'subscription_status', 'shipping_type',
       'discount_applied', 'promo_code_used', 'previous_purchases',
       'payment_method', 'frequency_of_purchases'],
      dtype='object')

In [16]:
df.rename(columns={'unnamed:_0': 'customer_id', 'purchase_amount_(usd)': 'purchase_amount'}, inplace=True)

In [17]:
#we now create a new column for grouping the ages, adult, young adult, middle aged  and senior
#the qcut here divides the age distribution into 4 equal parts
labels =['Young_adult', 'Adult', 'Middle_age', 'Senior']
df['age_group'] = pd.qcut(df['age'], q=4, labels =labels)

In [18]:
df[['age', 'age_group']].head(10)

,age,age_group
0,55,Middle_age
1,19,Young_adult
2,50,Middle_age
3,21,Young_adult
4,45,Middle_age
5,46,Middle_age
6,63,Senior
7,27,Young_adult
8,26,Young_adult
9,57,Middle_age


In [19]:
#We add anoher new column for purchase_frequency_days - tells us how often a customer shops- predictions for stocking up, which days are busy etc
frequency_mapping ={
    'Fortnightly':14,
    'Weekly':7,
    'Monthly':30,
    'Quarterly':90,
    'Bi-weekly':14,
    'Every three months':90,
    'Annually':365
}
df['purchase_frequency_days'] =df['frequency_of_purchases'].map(frequency_mapping)

In [35]:
df[['purchase_frequency_days', 'frequency_of_purchases']].head(10)

,purchase_frequency_days,frequency_of_purchases
0,14.0,Fortnightly
1,14.0,Fortnightly
2,7.0,Weekly
3,7.0,Weekly
4,365.0,Annually
5,7.0,Weekly
6,90.0,Quarterly
7,7.0,Weekly
8,365.0,Annually
9,90.0,Quarterly


In [20]:
#we note that the discount applied an the promo code used columns are the same and we dont need redundancy, so we go ahead and delete one 
#we first check if the two columns are same. If it returns true, then all the values in the two columns are same and we dont need duplicates
(df['discount_applied'] == df['promo_code_used']).all()

True

In [21]:
#we now drop one of the columns, we choose promo_code_used
df =df.drop('promo_code_used', axis =1)

In [22]:
df.columns

Index(['customer_id', 'age', 'gender', 'item_purchased', 'category',
       'purchase_amount', 'location', 'size', 'color', 'season',
       'review_rating', 'subscription_status', 'shipping_type',
       'discount_applied', 'previous_purchases', 'payment_method',
       'frequency_of_purchases', 'age_group', 'purchase_frequency_days'],
      dtype='object')

In [23]:
#we now connect to sql in this case using postgresql
!pip install psycopg2-binary sqlalchemy

In [34]:
#we now connect to postgrel
from sqlalchemy import create_engine
username ="postgres" #this is the default user
password = "muita"  #set up during installation
host = "localhost" #if its running locally which in this case it is
port = "5432" #default postgresql port
database = "Customer_Behavior" #the new database we created in pgAdmin

engine = create_engine(f"postgresql+psycopg2://{username}:{password}@{host}:{port}/{database}")

#we now load dataframe to postegresql
#table_name = "customer" #we choose any name for our table/scehma

df.to_sql(table_name, engine, if_exists ="replace", index = False)
print(f"Data Successfully loaded into table {table_name} in database {database}.")
                       


Data Successfully loaded into table customer in database Customer_Behavior.
